In [1]:
import os
import h5py
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
BASE_DIR = Path("/home/elias/data-transfer-student")

In [3]:
def _get_patch_count(h5_path: Path) -> int:
    with h5py.File(h5_path, "r") as f:
        return f["features"].shape[0]

def build_wsi_cohort(cohort_dir: Path) -> pd.DataFrame:
    df_clin = pd.read_csv(
        cohort_dir / "clinical/clinical_survival_per_slide.csv")
    wsi_dir = cohort_dir / "wsi_features/conch_v15"
    available_wsi = {p.stem for p in wsi_dir.glob("*.h5")}

    df_clin = df_clin.dropna(subset=['survival_months', 'censorship']).copy()
    df_clin = df_clin[df_clin['survival_months'] > 0].copy()
    df_clin['slide_id'] = df_clin['slide_id'].str.replace(
        '.svs', '', regex=False)
    df_clin = df_clin[df_clin["slide_id"].isin(available_wsi)].copy()

    
    df_clin["event"] = 1 - df_clin["censorship"] # 1 = Todesfall, 0 = zensiert


    dx_extracted = df_clin["slide_id"].str.extract(r"-DX(\d+)\.", expand=False)
    df_clin["dx_num"] = dx_extracted.fillna(999).astype(int)

    df_clin["n_patches"] = [
        _get_patch_count(wsi_dir / f"{s}.h5") for s in df_clin["slide_id"]
    ]
    
    df_clin = (
        df_clin.sort_values(by=["case_id", "dx_num", "n_patches"], ascending=[
                            True, True, False])
        .drop_duplicates(subset=["case_id"], keep="first")
        # da wir nur einen Slide pro Patient wollen, wird hier nach case_id gruppiert und der Slide mit der niedrigsten DX-Nummer und den meisten Patches behalten
        .reset_index(drop=True)
    )
    return df_clin

In [4]:
def compute_fold_time_bins(train_df: pd.DataFrame, val_df: pd.DataFrame, n_bins: int = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Berechnet die Quartil-Grenzen NUR auf den Todesfällen von train_df (verhindert Data Leakage)
    und wendet sie auf train_df und val_df an.
    """
    train_df = train_df.copy()
    val_df = val_df.copy()

    # 1. Fit nur auf den unzensierten Trainingsdaten dieses Folds
    uncensored_train = train_df.loc[train_df["event"] == 1, "survival_months"]
    _, q_bins = pd.qcut(uncensored_train, q=n_bins, retbins=True, labels=False)

    # 2. Äußere Grenzen auf +/- unendlich setzen, damit Val-Samples außerhalb
    #    der Trainings-Range sauber in Bin 0 bzw. Bin (n_bins - 1) fallen (keine NaNs!)
    q_bins[0] = -np.inf
    q_bins[-1] = np.inf

    # 3. Transform auf Train und Val mit denselben Trainings-Grenzen
    train_df["time_bin"] = pd.cut(
        train_df["survival_months"], bins=q_bins, labels=False, include_lowest=True
    ).astype(int)
    val_df["time_bin"] = pd.cut(
        val_df["survival_months"], bins=q_bins, labels=False, include_lowest=True
    ).astype(int)

    return train_df, val_df

In [ ]:
def cross_validate(df: pd.DataFrame, n_splits: int = 5, n_bins: int = 4, : int = 5) -> pd.DataFrame:
    best_split = None
    best_score = float("inf")

    # Berechnet die Zeitbins für die Stratifizierung
    uncensored_times = df.loc[df["event"] == 1, "survival_months"]
    _, q_bins = pd.qcut(uncensored_times, q=n_bins, retbins=True, labels=False)
    q_bins[0] = -np.inf
    q_bins[-1] = np.inf
    strat_bins = pd.cut(df["survival_months"], bins=q_bins, labels=False, include_lowest=True)
    strat_target = np.where(df["event"] == 1, strat_bins + 1, 0)

    # testen von verschiedenen Seeds für die Greedy-Gruppenzuweisung
    for seed in range(1000):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        
        folds = np.full(len(df), -1)
        valid_split = True
        event_counts = []
        fold_sizes = []

        for fold, (_, val_idx) in enumerate(sgkf.split(X=df, y=strat_target, groups=df["site"])): # fold ist die aktuelle Fold-Nummer, val_idx sind die Indizes der Validierungsdaten für diesen Fold
            folds[val_idx] = fold
            n_events = df.iloc[val_idx]["event"].sum()
            if n_events < min_events_per_fold:
                valid_split = False
                break
            event_counts.append(n_events)
            fold_sizes.append(len(val_idx))

        if valid_split:
            # Optimiere auf möglichst gleiche Anzahl an Events UND ähnliche Fold-Größe
            score = np.std(event_counts) * 2.0 + np.std(fold_sizes) * 0.5
            if score < best_score:
                best_score = score
                best_split = folds.copy()

    if best_split is None:
        raise RuntimeError(
            f"Kein gültiger Split mit mind. {min_events_per_fold} Events pro Fold gefunden! "
            "Eine einzelne 'site' dominiert die Kohorte zu stark."
        )

    df["fold"] = best_split
    
    # Kurzer Check-Print Log:
    print("Fold-Verteilung (Größe | Events):")
    for f in range(n_splits):
        sub = df[df["fold"] == f]
        print(f"  Fold {f}: n={len(sub):3d} | events={int(sub['event'].sum()):2d} | sites={sub['site'].nunique()}")
        
    return df

In [6]:
def build_dataset(cohort_dir: Path, n_bins: int = 4, n_splits: int = 5) -> pd.DataFrame:
    df_clin = build_wsi_cohort(cohort_dir)
    df_clin = cross_validate(df_clin, n_splits=n_splits, n_bins=n_bins)
    return df_clin

In [ ]:
def preload_wsi_features(df: pd.DataFrame, wsi_dir: Path) -> dict:
    """Lädt alle eindeutigen WSI-Feature-Dateien einmalig in den RAM."""
    wsi_dir = Path(wsi_dir)
    unique_slides = df['slide_id'].unique()
    cache = {}
    
    print(f"Lade {len(unique_slides)} WSI-Feature-Dateien in den RAM...")
    for slide_id in unique_slides:
        h5_path = wsi_dir / f"{slide_id}.h5"
        with h5py.File(h5_path, 'r') as f:
            # Direkt als float32-Tensor im RAM ablegen
            cache[slide_id] = torch.from_numpy(f['features'][:]).to(device, dtype=torch.float32) #device laedt die Daten auf die GPU, falls device='cuda', sonst auf die CPU
            
    ram_mb = sum(t.element_size() * t.nelement() for t in cache.values()) / (1024 ** 2)
    print(f"Fertig! Belegter RAM durch Feature-Cache: {ram_mb:.1f} MB")
    return cache


class SurvivalWSIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, wsi_dir: Path, max_patches: int = None, feature_cache: dict = None):
        self.df = df.reset_index(drop=True)
        self.wsi_dir = Path(wsi_dir)
        self.max_patches = max_patches
        self.feature_cache = feature_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        slide_id = row['slide_id']
        
        # 1. Aus dem RAM-Cache lesen (falls vorhanden), sonst Fallback auf SSD
        if self.feature_cache is not None and slide_id in self.feature_cache:
            features_tensor = self.feature_cache[slide_id]
        else:
            h5_path = self.wsi_dir / f"{slide_id}.h5"
            with h5py.File(h5_path, 'r') as f:
                features_tensor = torch.from_numpy(f['features'][:]).to(torch.float32)

        # 2. Optionales Subsampling direkt mit PyTorch auf dem Tensor
        if self.max_patches is not None and features_tensor.size(0) > self.max_patches:
            indices = torch.randperm(features_tensor.size(0))[:self.max_patches]
            features_tensor = features_tensor[indices]
        
        target = torch.tensor([int(row['time_bin']), int(row['censorship'])], dtype=torch.long)
        survival_months = torch.tensor(row['survival_months'], dtype=torch.float32)
        censorship = torch.tensor(row['censorship'], dtype=torch.float32)

        return features_tensor, target, survival_months, censorship


In [8]:
from sksurv.metrics import concordance_index_censored
from src.models.abmil import ABMIL

class NLLSurvLoss(nn.Module):
    """
    Discrete-Time Survival Loss (Negative Log-Likelihood).
    """
    def __init__(self, alpha: float = 0.15):
        super().__init__()
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # target: [B, 2] -> Spalte 0 = time_bin, Spalte 1 = censorship (1 = zensiert, 0 = Event)
        y = target[:, 0].view(-1, 1)
        c = target[:, 1].view(-1, 1).float()

        hazards = torch.sigmoid(logits)
        S = torch.cumprod(1 - hazards, dim=1) # kumulatives Produkt: Tensor, bei dem jeder Eintrag die Wahrscheinlichkeit angibt, bis zu diesem spezifischen Zeitpunkt (und allen vorherigen) überlebt zu haben. 
        S_padded = torch.cat([torch.ones_like(c), S], dim=1)

        s_prev = torch.gather(S_padded, dim=1, index=y).clamp(min=1e-7)
        h_this = torch.gather(hazards, dim=1, index=y).clamp(min=1e-7)
        s_this = torch.gather(S_padded, dim=1, index=y + 1).clamp(min=1e-7)

        uncensored_loss = -(1 - c) * (torch.log(s_prev) + torch.log(h_this))
        censored_loss = -c * torch.log(s_this)

        loss = (1.0 - self.alpha) * censored_loss + uncensored_loss
        return loss.mean()


/home/elias/miniconda3/envs/mil/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/elias/miniconda3/envs/mil/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def train_one_epoch(model, loader, loss_fn, optimizer, device, grad_accum_steps=8):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    num_batches = len(loader)

    for step, (features, target, _, _) in enumerate(loader):
        features = features.to(device)  # [1, M, 768]
        target = target.to(device)      # [1, 2]

        rem_batches = num_batches - (step // grad_accum_steps) * grad_accum_steps
        actual_accum_steps = min(grad_accum_steps, rem_batches)

        results_dict, _ = model(h=features, loss_fn=loss_fn, label=target)
        loss = results_dict['loss'] / actual_accum_steps
        loss.backward()

        total_loss += results_dict['loss'].item() # Nur für Logging, nicht für Backpropagation

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == num_batches:
            optimizer.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_risk_scores = []
    all_censorships = []
    all_event_times = []

    for features, target, surv_months, censorship in loader:
        features = features.to(device)
        target = target.to(device)

        results_dict, _ = model(h=features, loss_fn=loss_fn, label=target)
        total_loss += results_dict['loss'].item()

        # Risk Score aus den 4 Survival-Bins berechnen
        hazards = torch.sigmoid(results_dict['logits'])
        survival = torch.cumprod(1 - hazards, dim=1) #Berechnet die kumulative Überlebenswahrscheinlichkeit über die Zeit, indem das Produkt von 1 - hazards gebildet wird.
        risk = -torch.sum(survival, dim=1) # Berechnet den Risikowert als negative Summe der Überlebenswahrscheinlichkeiten (höherer Wert = höheres Risiko)


        all_risk_scores.extend(risk.cpu().numpy())
        all_censorships.extend(censorship.numpy())
        all_event_times.extend(surv_months.numpy())

    # sksurv erwartet event_indicator = True für Todesfall (1 - censorship)
    event_indicators = (1 - np.array(all_censorships)).astype(bool)
    c_index = concordance_index_censored(
        event_indicators,
        np.array(all_event_times),
        np.array(all_risk_scores)
    )[0]

    return total_loss / len(loader), c_index, all_risk_scores

In [10]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

class ExperimentLogger:
    def __init__(self, base_dir: str, config: dict):
        self.config = config
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_name = f"{config['cohort']}_{config['model_type']}_{timestamp}"
        
        self.base_dir = Path(base_dir)
        self.run_dir = self.base_dir / run_name
        self.ckpt_dir = self.run_dir / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. Config als JSON speichern
        with open(self.run_dir / "config.json", "w") as f:
            json.dump(self.config, f, indent=4)
            
        self.epoch_logs = []
        self.oof_records = []

    def log_splits(self, df: pd.DataFrame):
        # 2. Exakte Splits und Bins sichern
        cols = ["case_id", "slide_id", "site", "survival_months", "censorship", "event", "time_bin", "fold"]
        available_cols = [c for c in cols if c in df.columns]
        df[available_cols].to_csv(self.run_dir / "dataset_splits.csv", index=False)

    def log_epoch(self, fold: int, epoch: int, train_loss: float, val_loss: float, val_c_index: float):
        # 3. Verlauf pro Epoche merken
        self.epoch_logs.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_c_index": val_c_index
        })

    def log_oof_predictions(self, val_df: pd.DataFrame, risk_scores: list, fold: int):
        # 4. Beste Validierungs-Vorhersagen dieses Folds für spätere Kaplan-Meier-Analysen speichern
        oof_df = val_df[["case_id", "slide_id", "survival_months", "censorship", "event", "time_bin"]].copy()
        oof_df["fold"] = fold
        oof_df["risk_score"] = risk_scores
        self.oof_records.append(oof_df)

    def finalize(self, fold_c_best: list, fold_c_loss: list = None, fold_c_last: list = None):
        pd.DataFrame(self.epoch_logs).to_csv(self.run_dir / "metrics_per_epoch.csv", index=False)
        if self.oof_records:
            pd.concat(self.oof_records, ignore_index=True).to_csv(self.run_dir / "oof_predictions.csv", index=False)
        
        summary = {
            "run_name": self.run_dir.name,
            "mean_c_index": round(float(np.mean(fold_c_best)), 4),
            "std_c_index": round(float(np.std(fold_c_best)), 4),
            "fold_c_indices": [round(float(x), 4) for x in fold_c_best],
        }
        if fold_c_loss is not None:
            summary["mean_c_at_best_loss"] = round(float(np.mean(fold_c_loss)), 4)
            summary["std_c_at_best_loss"] = round(float(np.std(fold_c_loss)), 4)
        if fold_c_last is not None:
            summary["mean_c_last_epoch"] = round(float(np.mean(fold_c_last)), 4)
            summary["std_c_last_epoch"] = round(float(np.std(fold_c_last)), 4)

        summary.update(self.config)
        
        with open(self.run_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=4)
            
        master_csv = self.base_dir / "experiment_summary.csv"
        summary_flat = {k: str(v) if isinstance(v, list) else v for k, v in summary.items()}
        df_row = pd.DataFrame([summary_flat])
        df_row.to_csv(master_csv, mode="a", header=not master_csv.exists(), index=False)
        
        print(f"\nAlle Ergebnisse archiviert unter: {self.run_dir}")

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Nutze Device: {device}")

Nutze Device: cuda


In [ ]:
import random
# 1. Zentrale Konfiguration definieren

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

config = {
    "cohort": "TCGA-GBMLGG",
    "model_type": "abmil",
    "in_dim": 768, # equal to CONCH v1.5 embedding dimension
    "embed_dim": 512,
    "num_fc_layers": 1,
    "dropout": 0.25,
    "attn_dim": 384,
    "gate": True,
    "num_classes": 4,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-5,
    "grad_accum_steps": 32, # Wie in Papern Mcat, Motcat
    "max_patches_train": None,
    "loss_alpha": 0.15,
    "seed": 85
}

seed_everything(config["seed"])

dataset = build_dataset(BASE_DIR / config["cohort"], n_bins=config["num_classes"], n_splits=5)

logger = ExperimentLogger(base_dir="./runs", config=config)
logger.log_splits(dataset)
WSI_DIR = BASE_DIR / config["cohort"] / "wsi_features/conch_v15"

# Einmalig in den System-RAM laden
wsi_cache = preload_wsi_features(dataset, WSI_DIR)

fold_results_best_c = []
fold_results_best_loss = []
fold_results_last = []

use_pin_memory = (device.type == "cuda")

for fold in range(5):
    print(f"\n==================== FOLD {fold} ====================")
    seed_everything(config["seed"] + fold)
    train_df = dataset[dataset['fold'] != fold]
    val_df = dataset[dataset['fold'] == fold]

    train_df, val_df = compute_fold_time_bins(
        train_df, val_df, n_bins=config["num_classes"]
    )
    # Cache an beide Datasets übergeben
    train_ds = SurvivalWSIDataset(
        train_df, wsi_dir=WSI_DIR, max_patches=config["max_patches_train"], feature_cache=wsi_cache
    )
    val_ds = SurvivalWSIDataset(
        val_df, wsi_dir=WSI_DIR, max_patches=None, feature_cache=wsi_cache
    )
    
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)
    
    model = ABMIL(
        in_dim=config["in_dim"],
        embed_dim=config["embed_dim"],
        num_fc_layers=config["num_fc_layers"],
        dropout=config["dropout"],
        attn_dim=config["attn_dim"],
        gate=config["gate"],
        num_classes=config["num_classes"]
    ).to(device)
    
    loss_fn = NLLSurvLoss(alpha=config["loss_alpha"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])
    
    best_c_index = -1.0
    best_val_loss = float("inf")
    c_index_at_best_loss = 0.0
    best_risk_scores = None
    
    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(
            model, train_loader, loss_fn, optimizer, device, grad_accum_steps=config["grad_accum_steps"]
        )
        val_loss, val_c_index, val_risk_scores = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        
        logger.log_epoch(fold, epoch + 1, train_loss, val_loss, val_c_index)
        print(f"Epoch {epoch+1:02d}/{config['epochs']:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val C-Index: {val_c_index:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            c_index_at_best_loss = val_c_index
            
        if val_c_index > best_c_index:
            best_c_index = val_c_index
            best_risk_scores = val_risk_scores
            torch.save(model.state_dict(), logger.ckpt_dir / f"best_model_fold_{fold}.pt")
            
    print(f"--> Fold {fold} | Best C-Index: {best_c_index:.4f} | C-Index @ Best Loss: {c_index_at_best_loss:.4f} | Last C-Index: {val_c_index:.4f}")
    logger.log_oof_predictions(val_df, best_risk_scores, fold)
    
    fold_results_best_c.append(best_c_index)
    fold_results_best_loss.append(c_index_at_best_loss)
    fold_results_last.append(val_c_index)
    # GESAMTERGEBNIS & ARCHIVIERUNG 
print("\n==================================================")
print(f"5-Fold CV ({config['cohort']}) - Best Val C-Index : {np.mean(fold_results_best_c):.4f} ± {np.std(fold_results_best_c):.4f}")
print(f"5-Fold CV ({config['cohort']}) - @ Best Val Loss  : {np.mean(fold_results_best_loss):.4f} ± {np.std(fold_results_best_loss):.4f}")
print(f"5-Fold CV ({config['cohort']}) - Last Epoch       : {np.mean(fold_results_last):.4f} ± {np.std(fold_results_last):.4f}")
logger.finalize(fold_results_best_c, fold_results_best_loss, fold_results_last)


Fold-Verteilung (Größe | Events):
  Fold 0: n=104 | events=51 | sites=4
  Fold 1: n=132 | events=36 | sites=7
  Fold 2: n=132 | events=46 | sites=7
  Fold 3: n=119 | events=24 | sites=10
  Fold 4: n= 81 | events=30 | sites=7
Lade 568 WSI-Feature-Dateien in den RAM...
Fertig! Belegter RAM durch Feature-Cache: 4307.7 MB

==================== FOLD 0 ====================
Epoch 01/20 | Train Loss: 1.0607 | Val Loss: 1.4780 | Val C-Index: 0.6063
Epoch 02/20 | Train Loss: 0.7768 | Val Loss: 1.1828 | Val C-Index: 0.6442
Epoch 03/20 | Train Loss: 0.6707 | Val Loss: 1.1382 | Val C-Index: 0.6332
Epoch 04/20 | Train Loss: 0.6431 | Val Loss: 1.1797 | Val C-Index: 0.6321
Epoch 05/20 | Train Loss: 0.5963 | Val Loss: 1.2383 | Val C-Index: 0.6409
Epoch 06/20 | Train Loss: 0.5738 | Val Loss: 1.1873 | Val C-Index: 0.6299
Epoch 07/20 | Train Loss: 0.5501 | Val Loss: 1.2544 | Val C-Index: 0.6336
Epoch 08/20 | Train Loss: 0.5347 | Val Loss: 1.1764 | Val C-Index: 0.6317
Epoch 09/20 | Train Loss: 0.5107 | Val

In [13]:
torch.cuda.empty_cache()